# FastAPI: build and test a machine-learning API

Another application needs digit predictions without understanding the training code. Treat this as a contract between a caller and a model: the image must have a known shape and range, the response must preserve meaning, and invalid inputs must fail predictably. Follow one request from JSON through validation to prediction.

A trained model becomes useful to other applications when it can receive a request and return a prediction through a stable interface. FastAPI helps define that interface, validate incoming data, document the endpoints, and serve requests with Python type hints.

In this project we will train a small scikit-learn digit classifier, save it, and build an API that serves single and batch predictions. We will test the routes in-process with `TestClient`, examine the generated OpenAPI schema, and finish with instructions for running a local server.

> **Prerequisites:** Use Python 3.10 or later with FastAPI, Uvicorn, HTTPX, scikit-learn, NumPy, and joblib installed. For example, install `fastapi`, `uvicorn[standard]`, `httpx`, `scikit-learn`, and `joblib` in the notebook's environment. This tutorial uses Pydantic 2, which is installed with current FastAPI releases.

## 1. Import the project tools

FastAPI defines the web application, Pydantic describes and validates request and response data, and scikit-learn supplies a compact model and bundled dataset. `TestClient` exercises the API without starting a separate server process, which makes endpoint checks easy to repeat in a notebook.

In [1]:
from pathlib import Path
import importlib.util
import sys

import joblib
import numpy as np
import sklearn
from fastapi import FastAPI
from fastapi.testclient import TestClient
from pydantic import BaseModel, Field, field_validator
from sklearn.datasets import load_digits
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import accuracy_score, classification_report
from sklearn.model_selection import train_test_split

print('scikit-learn:', sklearn.__version__)
print('FastAPI:', FastAPI.__module__)

scikit-learn: 1.6.1
FastAPI: fastapi.applications


## 2. Agree on the image a caller is allowed to send

Sketch one valid request before reading the schema. Ask what would happen if a caller supplied a normal photograph or values from 0 to 255. Matching the training input contract matters even when JSON parsing succeeds.

The built-in digits dataset contains grayscale images with 8 rows and 8 columns. Each pixel is an integer from 0 through 16, and the label is one of the digits 0 through 9. The API will accept one image as an 8-by-8 nested list and return the predicted digit, its confidence, and the full probability distribution.

An explicit contract matters: clients need to know the exact dimensions, value range, and response shape. The API will reject malformed images with an HTTP 422 validation response instead of passing ambiguous input to the model.

In [2]:
digits = load_digits()
features = digits.images.reshape(len(digits.images), -1).astype(np.float32)
labels = digits.target.astype(np.int64)

X_train, X_test, y_train, y_test = train_test_split(
    features, labels, test_size=0.2, stratify=labels, random_state=42
)

print('Training examples:', len(X_train))
print('Test examples:', len(X_test))
print('Feature count per image:', X_train.shape[1])
print('Pixel range:', features.min(), 'to', features.max())
print('Classes:', np.unique(labels))

Training examples: 1437
Test examples: 360
Feature count per image: 64
Pixel range: 0.0 to 16.0
Classes: [0 1 2 3 4 5 6 7 8 9]


## 3. Train and evaluate a baseline model

The API should wrap a model that has already been trained and evaluated. We use a random forest here because it handles the small numeric feature set without image-specific preprocessing. The test split estimates performance on examples not used for fitting; it is not used to tune the model.

In [3]:
classifier = RandomForestClassifier(
    n_estimators=200,
    class_weight='balanced_subsample',
    random_state=42,
    n_jobs=-1,
)
classifier.fit(X_train, y_train)
test_predictions = classifier.predict(X_test)
test_accuracy = accuracy_score(y_test, test_predictions)

print(f'Test accuracy: {test_accuracy:.3f}')
print(classification_report(y_test, test_predictions, zero_division=0))

Test accuracy: 0.967
              precision    recall  f1-score   support

           0       0.97      0.97      0.97        36
           1       0.92      1.00      0.96        36
           2       1.00      0.97      0.99        35
           3       0.97      0.97      0.97        37
           4       0.97      1.00      0.99        36
           5       1.00      1.00      1.00        37
           6       1.00      0.97      0.99        36
           7       0.92      1.00      0.96        36
           8       0.94      0.86      0.90        35
           9       0.97      0.92      0.94        36

    accuracy                           0.97       360
   macro avg       0.97      0.97      0.97       360
weighted avg       0.97      0.97      0.97       360



## 4. Save the trained model artifact

The API process loads a saved model at startup. This separates model training from request handling: a deployed service does not retrain each time it starts. `joblib` is convenient for scikit-learn estimators, but only load model files from trusted sources because pickle-based formats can execute code during loading.

In [4]:
project_dir = Path.cwd()
output_dir = project_dir / 'outputs'
output_dir.mkdir(exist_ok=True)
model_path = output_dir / 'digits_random_forest.joblib'
joblib.dump(classifier, model_path)
print('Saved model:', model_path.resolve())

Saved model: /Users/anuradhakumari/Library/Mobile Documents/com~apple~CloudDocs/GitHub/Libraries/outputs/digits_random_forest.joblib


## 5. Design validated request and response models

Pydantic models define the JSON contract. Basic Python types validate that the body is shaped like a list of rows; a field validator enforces exactly 8 rows, exactly 8 values per row, and pixel values in the allowed range. FastAPI turns validation failures into structured 422 responses before the prediction function runs.

The response model documents the output and checks that the service returns a valid digit, confidence, and ten class probabilities.

In [5]:
service_source = """from pathlib import Path

import joblib
import numpy as np
from fastapi import FastAPI
from pydantic import BaseModel, Field, field_validator

BASE_DIR = Path(__file__).resolve().parent
MODEL_PATH = BASE_DIR / 'outputs' / 'digits_random_forest.joblib'
classifier = joblib.load(MODEL_PATH)

app = FastAPI(
    title='Handwritten Digit Classifier API',
    version='1.0.0',
    description='Predict a digit from an 8-by-8 grayscale image.',
)

class DigitRequest(BaseModel):
    image: list[list[int]]

    @field_validator('image')
    @classmethod
    def validate_image(cls, image):
        if len(image) != 8 or any(len(row) != 8 for row in image):
            raise ValueError('image must contain exactly 8 rows of 8 pixels')
        if any(pixel < 0 or pixel > 16 for row in image for pixel in row):
            raise ValueError('pixel values must be between 0 and 16')
        return image

class DigitResponse(BaseModel):
    digit: int = Field(ge=0, le=9)
    confidence: float = Field(ge=0.0, le=1.0)
    probabilities: list[float]

class BatchRequest(BaseModel):
    images: list[DigitRequest]

    @field_validator('images')
    @classmethod
    def validate_batch_size(cls, images):
        if not 1 <= len(images) <= 32:
            raise ValueError('batch must contain between 1 and 32 images')
        return images

class BatchResponse(BaseModel):
    predictions: list[DigitResponse]

def predict_image(image):
    row = np.asarray(image, dtype=np.float32).reshape(1, -1)
    probabilities = classifier.predict_proba(row)[0]
    digit = int(np.argmax(probabilities))
    return {
        'digit': digit,
        'confidence': float(probabilities[digit]),
        'probabilities': [float(value) for value in probabilities],
    }

@app.get('/health')
def health_check():
    return {'status': 'ok', 'model_loaded': classifier is not None}

@app.get('/model')
def model_info():
    return {
        'model_type': classifier.__class__.__name__,
        'input_shape': [8, 8],
        'pixel_range': [0, 16],
        'classes': [int(value) for value in classifier.classes_],
    }

@app.post('/predict', response_model=DigitResponse)
def predict(request: DigitRequest):
    return predict_image(request.image)

@app.post('/predict/batch', response_model=BatchResponse)
def predict_batch(request: BatchRequest):
    return {'predictions': [predict_image(item.image) for item in request.images]}
"""

service_path = project_dir / 'digit_api.py'
service_path.write_text(service_source, encoding='utf-8')
print('Wrote service module:', service_path.resolve())

Wrote service module: /Users/anuradhakumari/Library/Mobile Documents/com~apple~CloudDocs/GitHub/Libraries/digit_api.py


## 6. Load the service and check its health

The generated module loads the model once when the service starts, then reuses it for requests. `TestClient` calls the app directly without binding a network port. Check the health route and model metadata before testing predictions.

In [6]:
module_spec = importlib.util.spec_from_file_location('digit_api', service_path)
if module_spec is None or module_spec.loader is None:
    raise RuntimeError('Could not load the generated API module')
digit_api = importlib.util.module_from_spec(module_spec)
sys.modules[module_spec.name] = digit_api
module_spec.loader.exec_module(digit_api)
client = TestClient(digit_api.app)

health_response = client.get('/health')
model_response = client.get('/model')
print('Health:', health_response.status_code, health_response.json())
print('Model:', model_response.status_code, model_response.json())

Health: 200 {'status': 'ok', 'model_loaded': True}
Model: 200 {'model_type': 'RandomForestClassifier', 'input_shape': [8, 8], 'pixel_range': [0, 16], 'classes': [0, 1, 2, 3, 4, 5, 6, 7, 8, 9]}


## 7. Check what a successful response actually promises

Predict the probability vector length and its sum, then inspect the response. A 200 status verifies successful handling; it does not establish that the predicted digit is correct or the confidence is calibrated.

The request body is ordinary JSON. A successful response includes the predicted digit, its confidence, and one probability per class. Random-forest probabilities are useful scores, but they are not guaranteed to be calibrated measures of real-world certainty.

In [7]:
sample_index = 0
sample_image = digits.images[sample_index].astype(int).tolist()
prediction_response = client.post('/predict', json={'image': sample_image})
prediction = prediction_response.json()

print('HTTP status:', prediction_response.status_code)
print('True digit:', int(digits.target[sample_index]))
print('API response:', prediction)
print('Probability count:', len(prediction['probabilities']))

HTTP status: 200
True digit: 0
API response: {'digit': 0, 'confidence': 0.995, 'probabilities': [0.995, 0.0, 0.0, 0.0, 0.005, 0.0, 0.0, 0.0, 0.0, 0.0]}
Probability count: 10


## 8. Reject the wrong shape before it reaches the model

Predict the HTTP status for each invalid image. After running the requests, inspect both the status and error details. Distinguish a client input error from an unexpected server failure.

Validation is part of the public API contract. These requests contain a 7-by-8 image and an image with an out-of-range pixel. FastAPI should reject each request with HTTP 422 and useful details, rather than letting invalid input reach scikit-learn.

In [8]:
wrong_shape = client.post('/predict', json={'image': [[0] * 8 for _ in range(7)]})
wrong_range_image = [row[:] for row in sample_image]
wrong_range_image[0][0] = 17
wrong_range = client.post('/predict', json={'image': wrong_range_image})

print('Wrong shape:', wrong_shape.status_code, wrong_shape.json()['detail'])
print('Out-of-range pixel:', wrong_range.status_code, wrong_range.json()['detail'])
assert wrong_shape.status_code == 422
assert wrong_range.status_code == 422

Wrong shape: 422 [{'type': 'value_error', 'loc': ['body', 'image'], 'msg': 'Value error, image must contain exactly 8 rows of 8 pixels', 'input': [[0, 0, 0, 0, 0, 0, 0, 0], [0, 0, 0, 0, 0, 0, 0, 0], [0, 0, 0, 0, 0, 0, 0, 0], [0, 0, 0, 0, 0, 0, 0, 0], [0, 0, 0, 0, 0, 0, 0, 0], [0, 0, 0, 0, 0, 0, 0, 0], [0, 0, 0, 0, 0, 0, 0, 0]], 'ctx': {'error': {}}}]
Out-of-range pixel: 422 [{'type': 'value_error', 'loc': ['body', 'image'], 'msg': 'Value error, pixel values must be between 0 and 16', 'input': [[17, 0, 5, 13, 9, 1, 0, 0], [0, 0, 13, 15, 10, 15, 5, 0], [0, 3, 15, 2, 0, 11, 8, 0], [0, 4, 12, 0, 0, 8, 8, 0], [0, 5, 8, 0, 0, 9, 8, 0], [0, 4, 11, 0, 1, 12, 7, 0], [0, 2, 14, 5, 10, 12, 0, 0], [0, 0, 6, 13, 10, 0, 0, 0]], 'ctx': {'error': {}}}]


## 9. Preserve the identity of each image in a batch

Before sending several images, decide how a caller matches each response to its input. Check output count and order. A batch that returns plausible digits in the wrong order is still incorrect.

Batching reduces per-request overhead when a client needs several predictions. The request model validates every image and caps each batch at 32 examples. The endpoint returns one result per input image in the same order.

In [10]:
batch_images = [image.astype(int).tolist() for image in digits.images[:4]]
batch_payload = [{'image': image} for image in batch_images]
batch_response = client.post('/predict/batch', json={'images': batch_payload})
batch_result = batch_response.json()
print('HTTP status:', batch_response.status_code)
print('Predicted digits:', [item['digit'] for item in batch_result['predictions']])
print('Expected digits:', digits.target[:4].tolist())
assert batch_response.status_code == 200
assert len(batch_result['predictions']) == len(batch_images)

HTTP status: 200
Predicted digits: [0, 1, 2, 3]
Expected digits: [0, 1, 2, 3]


## 10. Inspect the generated API documentation

FastAPI generates an OpenAPI schema from routes, type annotations, and Pydantic models. Interactive Swagger UI is available at `/docs`, ReDoc at `/redoc`, and the machine-readable schema at `/openapi.json`. These descriptions help client developers use the API without reading its implementation.

In [11]:
openapi_response = client.get('/openapi.json')
openapi_schema = openapi_response.json()
print('OpenAPI version:', openapi_schema['openapi'])
print('Available paths:', sorted(openapi_schema['paths']))
print('Request schema:', openapi_schema['components']['schemas']['DigitRequest'])

OpenAPI version: 3.1.0
Available paths: ['/health', '/model', '/predict', '/predict/batch']
Request schema: {'properties': {'image': {'items': {'items': {'type': 'integer'}, 'type': 'array'}, 'type': 'array', 'title': 'Image'}}, 'type': 'object', 'required': ['image'], 'title': 'DigitRequest'}


## 11. Run the API as a local server

The notebook tests do not start a server. After running the cells above, open a terminal in the workspace directory and launch the generated service:

```bash
uvicorn digit_api:app --reload
```

Open `http://127.0.0.1:8000/docs` for interactive documentation. The model must remain at `outputs/digits_random_forest.joblib` relative to `digit_api.py`. Stop the server with Ctrl+C.

A request uses the same JSON contract:

```bash
curl -X POST http://127.0.0.1:8000/predict \
  -H 'Content-Type: application/json' \
  -d '{"image": [[0,0,0,0,0,0,0,0],[0,0,0,0,0,0,0,0],[0,0,0,0,0,0,0,0],[0,0,0,0,0,0,0,0],[0,0,0,0,0,0,0,0],[0,0,0,0,0,0,0,0],[0,0,0,0,0,0,0,0],[0,0,0,0,0,0,0,0]]}'
```

## 12. Production considerations

This is a local learning service, not a production deployment. Before exposing a model publicly, plan for authentication, authorization, rate limits, payload-size limits, timeouts, and structured logging. Pin dependencies and version model artifacts; monitor latency, error rates, input drift, and model quality. Avoid logging request bodies by default because they may contain sensitive data. Model probabilities may need calibration before they drive consequential decisions.

For larger workloads, load the model once per worker, benchmark memory and concurrency, and choose worker counts based on measured capacity rather than assuming more workers are always better.

## Recap and next experiments

This project trained and evaluated a model, saved the artifact, defined request and response schemas, validated single and batch inputs, tested success and failure paths, and inspected generated OpenAPI documentation.

Next, add structured request IDs without recording image contents, expose the model version, compare batch with single-request latency, or replace the estimator while keeping the HTTP contract stable. Add contract tests before changing that contract or deploying the service.

## Explain what the evidence supports

Describe a request that is valid JSON but invalid model input, and a valid input that can still receive a wrong prediction. Explain which evidence tests the service contract and which tests model quality.

Change one assumption or input at a time. Predict the effect before rerunning, compare it with the actual result, and explain any difference. Keep the original result available so that the comparison is meaningful.

## Second project: serve a bounded scoring batch with predictable errors

A client submits feature rows with duplicate IDs, invalid values, unexpected fields and oversized batches. Build a separate in-notebook application that validates the contract, invokes a trained classifier once per batch, preserves ID ordering and returns structured errors. Compare model-call counts for batched versus individual requests. TestClient keeps the exercise local; no service is published.

Run these cells in order. This extension uses its own `p2_` variables and writes only its named outputs under `outputs/project2/`. Predict each comparison before running it; the worked discussion after the code explains the result and its limits.

### Work through coercion at an API boundary

If an endpoint accepts `"0.5"` as a float silently, callers may never notice a serialization bug. Decide whether conversion is allowed. Here features require finite strict numbers, extra fields are forbidden, IDs must be unique within a batch, and a batch holds at most 16 rows. Correct JSON syntax alone is not sufficient input validation.

In [ ]:
from pathlib import Path
import numpy as np
import pandas as pd
import time
from IPython.display import display
p2_out=Path('outputs/project2');p2_out.mkdir(parents=True,exist_ok=True)

from fastapi import FastAPI
from fastapi.testclient import TestClient
from pydantic import BaseModel,ConfigDict,Field,FiniteFloat,model_validator
from typing import Annotated
from sklearn.datasets import make_classification
from sklearn.linear_model import LogisticRegression
p2_x,p2_y=make_classification(n_samples=300,n_features=2,n_informative=2,n_redundant=0,random_state=3)
p2_estimator=LogisticRegression().fit(p2_x,p2_y)
class P2Row(BaseModel):
    model_config=ConfigDict(extra='forbid')
    request_id:Annotated[str,Field(min_length=1,max_length=40)]
    features:Annotated[list[Annotated[FiniteFloat,Field(strict=True)]],Field(min_length=2,max_length=2)]
class P2Batch(BaseModel):
    model_config=ConfigDict(extra='forbid')
    rows:Annotated[list[P2Row],Field(min_length=1,max_length=16)]
    @model_validator(mode='after')
    def unique(self):
        ids=[r.request_id for r in self.rows]
        if len(ids)!=len(set(ids)):raise ValueError('request IDs must be unique within a batch')
        return self
p2_app=FastAPI();p2_inference_calls=0
@p2_app.post('/score')
def p2_score(payload:P2Batch):
    global p2_inference_calls
    p2_inference_calls+=1
    probabilities=p2_estimator.predict_proba([r.features for r in payload.rows])[:,1]
    return {'model_version':'teaching-v1','results':[{'request_id':r.request_id,'score':float(s)} for r,s in zip(payload.rows,probabilities)]}

### Work through batching before making a speed claim

Sixteen individual HTTP requests invoke the estimator sixteen times; one batch invokes it once. Verify numerical equality and ID order, then report the call reduction. TestClient latency is not production HTTP latency: it excludes a real network and deployment stack. The 16-row cap bounds work per accepted request, not the total number of simultaneous clients.

In [ ]:
p2_rows=[{'request_id':f'r{i}','features':p2_x[i].tolist()} for i in range(16)]
with TestClient(p2_app) as p2_client:
    p2_batch_response=p2_client.post('/score',json={'rows':p2_rows});assert p2_batch_response.status_code==200
    assert p2_inference_calls==1
    p2_batch_results=p2_batch_response.json()['results']
    p2_single=[p2_client.post('/score',json={'rows':[r]}).json()['results'][0] for r in p2_rows]
    assert p2_inference_calls==17
    assert [r['request_id'] for r in p2_batch_results]==[r['request_id'] for r in p2_rows]
    assert np.allclose([r['score'] for r in p2_single],[r['score'] for r in p2_batch_results])
    p2_invalid=[{'rows':p2_rows+[{'request_id':'overflow','features':[0.,0.]}]},
      {'rows':[p2_rows[0],p2_rows[0]]},{'rows':[{'request_id':'x','features':['0.5',1.0]}]},
      {'rows':[{'request_id':'x','features':[0.5,1.0],'extra':True}]}]
    for p2_payload in p2_invalid:
        p2_result=p2_client.post('/score',json=p2_payload)
        assert p2_result.status_code==422
    assert p2_inference_calls==17 # Rejected requests never reached the estimator.
pd.DataFrame(p2_batch_results).to_csv(p2_out/'fastapi_batch_scores.csv',index=False)
print({'batch_model_calls':1,'individual_model_calls':16,'invalid_requests_rejected':4})

### Explain what was verified

The checks establish input rejection, ordering and equivalent scoring for the same rows. They do not establish accuracy, calibration, access control, concurrency safety or service throughput. The classifier uses synthetic data only to make the interface concrete. A deployed API also needs an explicit response schema, request-size controls and an operational plan.

**Try:** reverse the batch input order. **Worked reasoning:** predictions should follow the reversed IDs rather than remain tied to old positions. If results are stored asynchronously, stable IDs become essential. Adding `async def` around blocking model work does not automatically create faster parallel inference; measure the actual worker/thread model.